# 🟡 بيقش — تدريب نموذج التعرف على العملة

دفتر جاهز يشغّل خط المعالجة المُصَحَّح كاملاً: تكثير بلا تسريب ← تدريب بهندسة موحّدة ← تصدير `bigash_v18.tflite`.

**مهم:** من قائمة `Runtime` ← `Change runtime type` اختر **GPU** قبل البدء، ثم شغّل الخلايا بالترتيب من الأعلى للأسفل.

## ١) التحقق من البيئة

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus if gpus else 'لا يوجد GPU — افتح Runtime > Change runtime type > GPU')

## ٢) رفع سكربتات المعالجة المُصَحَّحة

شغّل الخلية التالية واختر هذين الملفين من مجلد `tools/` في مشروعك:
`augment_dataset.py` و `train_bigash_model.py`.

In [ ]:
import os
from google.colab import files
os.makedirs('tools', exist_ok=True)
print('اختر: augment_dataset.py و train_bigash_model.py')
uploaded = files.upload()
for name in list(uploaded):
    os.replace(name, os.path.join('tools', name))
    print('رُفع:', name)
with open('tools/augment_dataset.py', encoding='utf-8') as f:
    assert 'LEAK-FREE SPLIT' in f.read(), 'نسخة قديمة! ارفع augment_dataset.py المُصَحَّح.'
print('السكربتات جاهزة — نسخة بلا تسريب.')

## ٣) ربط Google Drive ومجلد الصور

صورك يمكن أن تكون **ملف `.zip`** أو **مجلداً جاهزاً** على Drive. في الحالتين يجب أن تحوي مجلداً فرعياً لكل فئة بهذه الأسماء بالضبط:

```
other/  y50/  y100/  y200/  y250/  y500/  y1000/
```

الخلية التالية تفكّ ضغط الـ zip تلقائياً وتكتشف مجلد الفئات. عدّل `ZIP_PATH` إن لزم.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, pathlib, os

# ✏️ مسار صورك على Drive: ملف .zip أو مجلد بفئات العملة
ZIP_PATH = '/content/drive/MyDrive/bigash_backup/dataset_cleaned.zip'
EXTRACT_TO = '/content/dataset_raw'
EXPECTED = {'other', 'y50', 'y100', 'y200', 'y250', 'y500', 'y1000'}

if ZIP_PATH and os.path.exists(ZIP_PATH):
    if not os.path.exists(EXTRACT_TO):
        print('جارٍ فكّ الضغط ... (1-3 دقائق)')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(EXTRACT_TO)
        print('تم فكّ الضغط.')
    else:
        print('مفكوك مسبقاً.')
    search_root = EXTRACT_TO
else:
    # لو صورك مجلد جاهز وليست zip، ضع مساره هنا
    search_root = '/content/drive/MyDrive/bigash/dataset'

def is_class_dir(p):
    try:
        subs = {s.name for s in p.iterdir() if s.is_dir()}
    except Exception:
        return False
    return len(EXPECTED & subs) >= 4

DATASET_DIR = None
if is_class_dir(pathlib.Path(search_root)):
    DATASET_DIR = search_root
else:
    for d in pathlib.Path(search_root).rglob('*'):
        if d.is_dir() and is_class_dir(d):
            DATASET_DIR = str(d)
            break

print('DATASET_DIR =', DATASET_DIR)
assert DATASET_DIR, 'لم أجد مجلد الفئات — شغّل خلية فيها: !ls -R /content/dataset_raw | head -60'

In [ ]:
import pathlib
EXPECTED = {'other', 'y50', 'y100', 'y200', 'y250', 'y500', 'y1000'}
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
root = pathlib.Path(DATASET_DIR)
assert root.is_dir(), 'المجلد غير موجود: ' + str(DATASET_DIR)
found = {d.name for d in root.iterdir() if d.is_dir()}
total = 0
for c in sorted(found):
    n = len([p for p in (root / c).iterdir() if p.suffix.lower() in EXTS])
    total += n
    print(c, '->', n, '(قليل جداً — أضف المزيد)' if n < 30 else '')
print('الإجمالي:', total)
missing = EXPECTED - found
extra = found - EXPECTED
if missing:
    print('فئات ناقصة:', sorted(missing))
if extra:
    print('مجلدات بأسماء غير متوقعة:', sorted(extra))
if not missing and not extra:
    print('بنية المجلدات صحيحة.')

## ٤) التكثير + التقسيم الآمن (بلا تسريب)

يقسّم الصور الأصلية إلى تدريب/تحقق **قبل** التكثير، ويُبقي صور التحقق خاماً.

In [ ]:
!python tools/augment_dataset.py --input_dir "{DATASET_DIR}" --output_dir /content/dataset_aug --variants 12 --val_split 0.2

## ٥) التدريب

قد يستغرق 20–60 دقيقة حسب حجم بياناتك. راقب `val_accuracy` في المخرجات — هذه أوّل مرة يكون رقماً صادقاً.

لتجربة دقة أعلى (أبطأ): غيّر `mobilenetv2` إلى `efficientnetb0`.

In [ ]:
!python tools/train_bigash_model.py --data_dir /content/dataset_aug --output_dir /content/out --backbone mobilenetv2

## ٦) مراجعة المخرجات

السكربت يطبع الدقة النهائية ومصفوفة الالتباس أعلاه. الخلية التالية تتأكد من إنشاء الملفات.

In [ ]:
import pathlib
out = pathlib.Path('/content/out')
for f in ['bigash_v18.tflite', 'labels.txt']:
    p = out / f
    print(f, '->', str(p.stat().st_size // 1024) + ' KB' if p.exists() else 'مفقود!')
print()
print('labels.txt:')
print(pathlib.Path('/content/out/labels.txt').read_text(encoding='utf-8'))

## ٧) التنزيل ووضع الملفات في التطبيق

In [ ]:
import shutil, pathlib
dst = pathlib.Path('/content/drive/MyDrive/bigash')
dst.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/out/bigash_v18.tflite', dst / 'bigash_v18.tflite')
shutil.copy('/content/out/labels.txt', dst / 'labels.txt')
print('نُسخت نسخة احتياطية إلى:', dst)

from google.colab import files
files.download('/content/out/bigash_v18.tflite')
files.download('/content/out/labels.txt')

### وضع الملفات في التطبيق

1. انسخ `bigash_v18.tflite` و `labels.txt` إلى مجلد `assets/` (استبدل القديمة).
2. شغّل: `flutter clean` ثم `flutter pub get` ثم `flutter build apk --release`.
3. اختبر على عملات حقيقية بإضاءات وزوايا وخلفيات مختلفة.

### إذا بقيت الدقة ضعيفة

- `val_accuracy` منخفض ⇐ البيانات غير كافية: أضف صوراً حقيقية أكثر لكل فئة (هذا أهم عامل).
- راجع مصفوفة الالتباس لمعرفة أي فئتين يخلط النموذج بينهما، وزِد صورهما.